> **Historical notebook: supervised DiT / Euclidean Flow Matching (L=32).**
> This notebook evaluates the earlier model trained on Wolff configurations. Its DiT includes the historical VorticityExtractor; it is not the current data-free SPS implementation. The current SPS uses a separate periodic CNN and stochastic paths.
> The original scientific code is retained, with saved outputs removed for publication. A clean clone does not contain `checkpoints/best_dit_flow.pth`, the Wolff reference tensors, or the saved raw comparison arrays. The legacy comparison can be regenerated only after its required data and checkpoint are supplied; a newly trained checkpoint is a new experiment.
> Read the [historical baseline guide](../docs/public/legacy.md) for the saved diagnostic and its limits.


# 02 — Flow Matching 采样与物理验证

本 Notebook 对训练好的 **Physics-Informed DiT** (Flow Matching) 模型进行端到端验证：

1. **模型加载** — 实例化 `PhysicsInformedDiT` 并加载 `checkpoints/best_dit_flow.pth`
2. **欧拉求解器** — 从高斯噪声 $x_0 \sim \mathcal{N}(0, I)$ 出发，沿学习到的速度场积分 ODE 至 $t=1$
3. **物理还原** — 将生成的 $(\cos\theta, \sin\theta)$ 双通道还原为角度矩阵，并计算拓扑涡旋荷
4. **可视化** — HSV 色轮 + 矢量场 + 涡旋/反涡旋标记（学术出版级，全英文标签）

**前置依赖**: 需先运行 `src/train_flow.py` 生成 checkpoint。


当前用途：欧氏 Flow Matching 基线诊断，还不是 SPS。2026-09-21 已将混合旧输出归档并清理；本次整理不改变模型权重或采样算法。关联指数是有限尺寸、指定拟合窗口下的描述量，不能单独判定模式坍缩。


## 1. 环境准备与路径设置

In [ ]:
from pathlib import Path

# Find the project from either the repository root or notebooks/.
PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / 'src' / 'model_dit.py').is_file() and (p / 'data').is_dir()
)
FIGURE_DIR = PROJECT_ROOT / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

import os
import sys
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

%matplotlib inline

# 动态将 ../src 添加到 sys.path (支持 notebooks/ 与项目根目录两种运行位置)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

torch.manual_seed(20260921)  # reproducible diagnostic sampling

from model_dit import PhysicsInformedDiT

L = 32  # 晶格尺寸
print(f'Python env ready. PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')


## 2. 模型加载

实例化网络并加载训练得到的最佳权重，切换到 `eval()` 模式。

In [ ]:
# 设备检测 (CUDA 优先)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# 实例化 Physics-Informed DiT
model = PhysicsInformedDiT().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')

# 鲁棒 checkpoint 路径检测 (支持 notebooks/ 与项目根目录两种运行位置)
ckpt_path = PROJECT_ROOT / 'checkpoints' / 'best_dit_flow.pth'
if not ckpt_path.is_file():
    raise FileNotFoundError(f'Missing checkpoint: {ckpt_path}')

# 加载权重 (兼容不同 PyTorch 版本的 weights_only 参数)
try:
    state_dict = torch.load(ckpt_path, map_location=device, weights_only=False)
except TypeError:
    state_dict = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

print(f'Checkpoint loaded: {os.path.abspath(ckpt_path)}')
print('Model set to eval() mode.')


## 3. 欧拉求解器 (Euler ODE Solver)

Flow Matching 定义了一条从噪声到数据的概率流 ODE：

$$\frac{dx_t}{dt} = v_\theta(x_t, t), \quad x_0 \sim \mathcal{N}(0, I)$$

用欧拉法以步长 $dt = 1/N_{steps}$ 从 $t=0$ 积分到 $t=1$，终点即为生成构型。

In [ ]:
def sample_flow(model, device, num_steps=50):
    """Euler ODE solver for Flow Matching sampling.

    Integrates dx/dt = v_theta(x_t, t) from t=0 (Gaussian noise)
    to t=1 (generated physical configuration).

    Args:
        model:     trained PhysicsInformedDiT (velocity field)
        device:    target device
        num_steps: number of Euler integration steps
    Returns:
        x: (1, 2, L, L) generated (cos, sin) tensor at t=1
    """
    model.eval()

    # t=0 起点: 高斯白噪声
    x = torch.randn(1, 2, 32, 32, device=device)

    # 时间步长
    dt = 1.0 / num_steps

    with torch.no_grad():
        for step in range(num_steps):
            t_val = step * dt                            # t ∈ [0, 1-dt]
            t = torch.full((1,), t_val, device=device)   # 标量 t → (1,) 张量
            v = model(x, t)                              # 速度场 v(x_t, t)
            x = x + v * dt                               # Euler 更新

    return x


# 生成一个样本
gen = sample_flow(model, device, num_steps=50)
print(f'Generated tensor shape: {tuple(gen.shape)}')
print(f'Value range: [{gen.min().item():.4f}, {gen.max().item():.4f}]')
print(f'|cos| L2 norm: {gen[0, 0].norm().item():.2f} | |sin| L2 norm: {gen[0, 1].norm().item():.2f}')


## 4. 特征还原与涡旋计算

- 使用 `torch.atan2` 将双通道 $(\cos\theta, \sin\theta)$ 还原为角度矩阵 $\theta$
- 复用 01 号 Notebook 的 `calculate_vorticity`：沿每个 Plaquette 闭合回路累计相位差
  （wrap 到 $[-\pi, \pi)$），除以 $2\pi$ 得到拓扑涡旋荷

In [ ]:
def calculate_vorticity(theta):
    """Compute the Plaquette vorticity of the 2D XY model.

    Args:
        theta: (L, L) angle matrix
    Returns:
        vorticity: (L, L) integer array, +1=vortex, -1=anti-vortex, 0=none
    """
    L = theta.shape[0]
    vorticity = np.zeros((L, L), dtype=int)

    for i in range(L):
        for j in range(L):
            ip1 = (i + 1) % L  # 周期性边界 (PBC)
            jp1 = (j + 1) % L

            # 回路: (i,j) → (i+1,j) → (i+1,j+1) → (i,j+1) → (i,j)
            diffs = [
                theta[ip1, j]    - theta[i, j],       # (i,j) → (i+1,j)
                theta[ip1, jp1]  - theta[ip1, j],     # (i+1,j) → (i+1,j+1)
                theta[i, jp1]   - theta[ip1, jp1],   # (i+1,j+1) → (i,j+1)
                theta[i, j]     - theta[i, jp1],      # (i,j+1) → (i,j)
            ]

            total = 0.0
            for d in diffs:
                # wrap 到 [-π, π)
                d_wrapped = (d + np.pi) % (2 * np.pi) - np.pi
                total += d_wrapped

            vorticity[i, j] = int(round(total / (2 * np.pi)))

    return vorticity


In [ ]:
# torch.atan2 还原角度矩阵: (1, 2, L, L) → (L, L)
theta = torch.atan2(gen[:, 1:2, :, :], gen[:, 0:1, :, :]).squeeze().cpu().numpy()

print(f'Angle matrix shape: {theta.shape}')
print(f'Angle range: [{theta.min():.4f}, {theta.max():.4f}]')
print(f'Theoretical range: [-π, π) = [-{np.pi:.4f}, {np.pi:.4f})')

# 计算涡旋荷
vorticity = calculate_vorticity(theta)

n_pos = int(np.sum(vorticity == 1))
n_neg = int(np.sum(vorticity == -1))
print(f'\nVortices (+1): {n_pos}')
print(f'Anti-vortices (-1): {n_neg}')
print(f'Net vorticity: {vorticity.sum()} (PBC identity; not a quality score)')


## 5. 高质量可视化

HSV 色轮热力图（周期角度）+ `quiver` 矢量场 + 红色/蓝色圆点标记涡旋/反涡旋
（位于 Plaquette 中心）。全英文标签，学术出版级标准。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

# --- 背景: 角度热力图 (HSV 色轮天然适合表示角度) ---
im = ax.imshow(theta, cmap='hsv', origin='lower',
               extent=[-0.5, L - 0.5, -0.5, L - 0.5],
               vmin=-np.pi, vmax=np.pi, interpolation='nearest')

# --- 矢量场: 自旋方向 (cos θ, sin θ) ---
x = np.arange(L)
y = np.arange(L)
X, Y = np.meshgrid(x, y)
U = np.cos(theta)
V = np.sin(theta)
ax.quiver(X, Y, U, V, color='black', scale=30, width=0.004,
          headwidth=3, headlength=4, pivot='middle')

# --- 涡旋标记 (位于 Plaquette 中心) ---
pos = np.argwhere(vorticity == 1)
neg = np.argwhere(vorticity == -1)

if len(pos) > 0:
    ax.plot(pos[:, 1] + 0.5, pos[:, 0] + 0.5, 'o',
            color='red', markersize=9, markeredgecolor='darkred',
            markeredgewidth=1.2, label=f'Vortex (+1) (n={n_pos})', zorder=5)
if len(neg) > 0:
    ax.plot(neg[:, 1] + 0.5, neg[:, 0] + 0.5, 'o',
            color='blue', markersize=9, markeredgecolor='darkblue',
            markeredgewidth=1.2, label=f'Anti-vortex (-1) (n={n_neg})', zorder=5)

# --- 装饰 ---
ax.set_xlim(-0.5, L - 0.5)
ax.set_ylim(-0.5, L - 0.5)
ax.set_xlabel('x', fontsize=13)
ax.set_ylabel('y', fontsize=13)
ax.set_title(f'Generated XY Model Spin Configuration (L={L}, Flow Matching)\n'
             f'Euler ODE, 50 steps | +1: {n_pos}  -1: {n_neg}', fontsize=14)
ax.legend(fontsize=11, loc='upper right', framealpha=0.9)

cbar = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('θ (rad)', fontsize=12)

plt.tight_layout()
plt.show()


## 6. 生成样本画廊

展示同一个模型在不同随机起点下的输出。此处的 8 张图用于直观观察；下一节保存的 128 个样本用于正式统计。
正负涡旋数相等主要来自周期边界，不能作为学对物理的证据。


In [ ]:
n_show = 8
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for k, ax in enumerate(axes.flat):
    # 采样 + 角度还原 + 涡旋统计
    g = sample_flow(model, device, num_steps=50)
    th = torch.atan2(g[:, 1:2, :, :], g[:, 0:1, :, :]).squeeze().cpu().numpy()
    vor = calculate_vorticity(th)
    cnt_pos = int((vor == 1).sum())
    cnt_neg = int((vor == -1).sum())

    # HSV 角度热力图
    ax.imshow(th, cmap='hsv', origin='lower',
              vmin=-np.pi, vmax=np.pi, interpolation='nearest')

    # 涡旋标记
    p = np.argwhere(vor == 1)
    n = np.argwhere(vor == -1)
    if len(p) > 0:
        ax.plot(p[:, 1] + 0.5, p[:, 0] + 0.5, 'o', color='red',
                markersize=6, markeredgecolor='darkred', markeredgewidth=1.0)
    if len(n) > 0:
        ax.plot(n[:, 1] + 0.5, n[:, 0] + 0.5, 'o', color='blue',
                markersize=6, markeredgecolor='darkblue', markeredgewidth=1.0)

    ax.set_title(f'Sample #{k} | +1: {cnt_pos}  -1: {cnt_neg}', fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle('Generated XY Configurations (Flow Matching, L=32, T≈0.89)',
             fontsize=14)
plt.tight_layout()
plt.show()


## 7. 与 Wolff 比较生成质量：统一的保存样本

本节与 notebook 03 共用 `results/fm_wolff_baseline/` 中的一次正式诊断。首次运行会生成并保存 128 个 FM 样本，以及 4 条新 Wolff 链、共 1024 个参考样本；以后读取同一批样本，不重新抽签。

比较温度为 β=1.12，即 T=0.892857…，来自原训练数据生成源码；旧数据文件名约写为 0.89，且没有内嵌完整温度元数据。03 的严格 T=0.89 扫描点保留为背景，不混作相同温度的主对照。

平均值的 95% 区间表示抽样不确定性：FM 按样本重采样；Wolff 按链内连续 16 个样本组成的块重采样。SD 则表示构型间波动。链的相关性检查会一并打印；这些有限检查不保证完全平衡。

`pair count = (N_plus + N_minus)/2` 是数量约定，未识别实际束缚对。本节不计算接受率。旧的不适用 IMH 估计保存在 archive 中。


In [ ]:
import math
import numpy as np
from xy_observables import energy_density, observables

# Energy and vortex statistics now share the implementation used by notebook 03.
# ---------------------------------------------------------------------------
# 2) Spin-spin correlation G(r) via FFT (fast for large ensembles)
# ---------------------------------------------------------------------------
def spin_correlation(theta, rmin=1.0, rmax=None):
    """Return (r, G(r)) for PBC distances.

    Uses the circular autocorrelation of z = exp(i theta), which gives
    <exp(i(theta_i - theta_j))> for each lattice displacement.  The result is
    binned by exact squared distance r^2 = dx^2 + dy^2.
    """
    theta = np.asarray(theta)
    single = (theta.ndim == 2)
    if single:
        theta = theta[None, ...]

    L = theta.shape[-1]
    z = np.exp(1j * theta)
    F = np.fft.fft2(z, axes=(-2, -1))
    C = np.fft.ifft2(F * np.conj(F), axes=(-2, -1)).real / (L * L)
    C = np.fft.fftshift(C, axes=(-2, -1))
    C = C.mean(axis=0)

    coords = np.arange(L) - L // 2
    X, Y = np.meshgrid(coords, coords, indexing='ij')
    r2 = X.astype(np.int64) ** 2 + Y.astype(np.int64) ** 2

    shells = sorted(set(r2[r2 > 0]))
    radii = np.sqrt(np.asarray(shells, dtype=float))
    G = np.asarray([C[r2 == q].mean() for q in shells])

    if rmax is None:
        rmax = L / math.sqrt(2.0)
    mask = (radii >= rmin) & (radii <= rmax)
    return radii[mask], G[mask]


def fit_power_law(r, G, rmin=2.0, rmax=None):
    """Fit log G = c - eta log r over positive correlation values."""
    r = np.asarray(r)
    G = np.asarray(G)
    if rmax is None:
        rmax = np.inf
    mask = (r >= rmin) & (r <= rmax) & (G > 0)
    slope, intercept = np.polyfit(np.log(r[mask]), np.log(G[mask]), 1)
    eta = -slope
    return eta, intercept, mask




In [ ]:
# One saved experiment supplies both the four-panel comparison and G(r).
import sys
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from compare_fm_wolff import ensure_run, plot_comparison, print_summary

report, saved = ensure_run()
print_summary(report)
gen_theta = saved['fm_theta']
ref_theta = saved['wolff_theta'].reshape(-1, L, L)
N_METRIC, N_REF = len(gen_theta), len(ref_theta)
E_gen, E_ref = energy_density(gen_theta), energy_density(ref_theta)

for quantity in ('pairs', 'energy'):
    checks = report['summaries'][quantity]['wolff_checks']
    print(quantity, 'Wolff split R-hat =', round(checks['split_rhat'], 4))
    print('  lag-one correlation by chain:', np.round(checks['lag1_by_chain'], 3))
    print('  SEM by block length:', checks['block_sem'])


### 7.1 同一张图：参考分布与模型偏差

- **A、B**：Wolff 多温度均值曲线及构型间的 SD 阴影；叠加目标温度下的 FM 与新 Wolff 均值及 95% 区间。
- **C、D**：相同目标温度下的数量和能量分布；纵轴是落入每个区间的构型比例，因此两组样本数不同也能比较。虚线标出均值。
- 涡旋越少或能量越低不代表越好，目标是分布接近参考。这张图诊断当前权重与 Euler 设置，不证明整个欧氏 FM 方法不可能成功。


In [ ]:
comparison = plot_comparison(report, saved)
from compare_fm_wolff import DEFAULT_OUTPUT
for extension in ('png', 'pdf'):
    comparison.savefig(DEFAULT_OUTPUT / f'comparison.{extension}', dpi=180)
plt.show()


### 7.2 同一批样本的空间关联

继续检查远距离自旋方向的相似程度。以下 G(r) 使用同一批 128 个 FM 与 1024 个新 Wolff 样本。η 只是有限尺寸和选定拟合区间内的描述，不能单独判定坍缩。

In [ ]:
# ---------------------------------------------------------------------------
# Spin correlation: log G(r) vs log r with a BKT eta = 0.25 reference line
# ---------------------------------------------------------------------------
r_gen, G_gen = spin_correlation(gen_theta)
r_ref, G_ref = spin_correlation(ref_theta)

# Automatic fit is only a first-pass estimate; use the plot below to choose
# a physically sensible fitting window by hand before trusting eta.
eta_gen, _, fit_gen = fit_power_law(r_gen, G_gen, rmin=2.0, rmax=L / 2.0)
eta_ref, _, fit_ref = fit_power_law(r_ref, G_ref, rmin=2.0, rmax=L / 2.0)
print(f'Flow Matching: eta = {eta_gen:.3f} (BKT expectation near 0.25)')
print(f'Wolff:        eta = {eta_ref:.3f}')

plt.figure(figsize=(8, 6))
plt.plot(np.log(r_gen), np.log(G_gen), 'o-', markersize=5,
         label=f'Flow Matching ($\eta$={eta_gen:.3f})')
plt.plot(np.log(r_ref), np.log(G_ref), 's-', markersize=5,
         label=f'Wolff ($\eta$={eta_ref:.3f})')

# Draw a dashed line with the exact BKT slope eta = 0.25.
# The intercept is anchored at the smallest plotted r using the geometric mean
# of the two curves, so it is a visual reference, not a fit.
r_anchor = max(float(r_gen[0]), float(r_ref[0]))
G_anchor = float(np.sqrt(G_gen[0] * G_ref[0])) if (G_gen[0] > 0 and G_ref[0] > 0) else 1.0
r_line = np.linspace(r_anchor, max(float(r_gen.max()), float(r_ref.max())), 100)
logG_line = np.log(G_anchor) - 0.25 * (np.log(r_line) - np.log(r_anchor))
plt.plot(np.log(r_line), logG_line, 'k--', linewidth=2,
         label=r'BKT reference: $\eta=0.25$')

plt.xlabel(r'$\log r$')
plt.ylabel(r'$\log G(r)$')
plt.title('Spin-Spin Correlation: Flow Matching vs Wolff (BKT)')
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend()
plt.tight_layout()

# Save the correlation check from exactly the same ensembles as the four-panel figure.
from compare_fm_wolff import DEFAULT_OUTPUT
plt.gcf().savefig(DEFAULT_OUTPUT / 'correlation.png', dpi=180, bbox_inches='tight')
plt.gcf().savefig(DEFAULT_OUTPUT / 'correlation.pdf', bbox_inches='tight')

plt.show()
